## 1.1 Знакомство с данными

Источник: Zingat — платформа объявлений о недвижимости.

Основные признаки:
- `type` — тип недвижимости (`Konut` = жильё).
- `sub_type` — подтип (`Daire` = квартира).
- `start_date`, `end_date` — даты начала и окончания активности объявления.
- `listing_type` — тип объявления (`Satılık` = продажа, `Kiralık` = аренда).
- `tom` — время на рынке (time on market).
- `building_age` — возраст здания. Значение `arası` означает «между».
- `total_floor_count` — общее количество этажей в здании.
- `room_count` — количество комнат, например `2+1` (2 комнаты + гостиная).
- `size` — площадь объекта, м².

Целевая переменная — цена объекта.  
Единицы измерения: площадь — м², цена — денежные единицы, время — дни.

In [ ]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42
DATA_PATH = "zingat_data.csv"
TARGET_CANDIDATES = ["price", "fiyat", "Price", "Fiyat", "target"]

In [ ]:
df = pd.read_csv("/home/wannabe/Desktop/works/python_machine/real_estate_data.csv")

print("Размерность:", df.shape)
print("\nСтолбцы:")
print(df.columns.tolist())

print("\nПервые строки:")
print(df.head())

print("\nПоследние строки:")
print(df.tail())

print("\nСлучайная строка:")
print(df.sample(1, random_state=RANDOM_STATE))

df.info()

На этом шаге определяется фактическая структура таблицы: число строк и столбцов, типы данных, наличие пропусков. Первые, последние и случайная строки показывают форматы значений: room_count может быть строкой 2+1, а building_age — содержать текстовые диапазоны.

In [ ]:
target = next((c for c in TARGET_CANDIDATES if c in df.columns), None)

if target is None:
    print("ВНИМАНИЕ: столбец с ценой не найден.")
    print("Доступные столбцы:", df.columns.tolist())
    print("Необходимо уточнить, какой столбец является целевым.")
else:
    print("Целевая переменная:", target)

Если целевая переменная не найдена автоматически, дальнейшее моделирование невозможно без уточнения. В словаре Zingat цена не описана, поэтому её наличие в файле нужно проверить отдельно.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()

print("Числовые признаки:", num_cols)
print("Категориальные признаки:", cat_cols)

print("\nОписательная статистика числовых признаков:")
print(df[num_cols].describe().T)

Числовые признаки описываются через минимум, максимум, среднее, медиану, квартили и стандартное отклонение. Это позволяет оценить масштаб значений и предварительно увидеть смещения. Например, size может иметь большой разброс, а tom — содержать нулевые или аномально большие значения

In [ ]:
for col in cat_cols:
    print(f"\n--- {col} ---")
    print("Уникальных значений:", df[col].nunique(dropna=False))
    print(df[col].value_counts(dropna=False).head(20))

Категориальные признаки показывают частоту категорий Для listing_type ожидаются значения Satılık и Kiralık Для sub_type — Daire и другие. Если категорий слишком много, потребуется группировка редких значений в «прочее» или целевое кодирование

In [ ]:
if target is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[target], kde=True, ax=axes[0])
    axes[0].set_title(f"Распределение {target}")
    sns.boxplot(x=df[target], ax=axes[1])
    axes[1].set_title(f"Ящик с усами: {target}")
    plt.tight_layout()
    plt.show()

    print("Асимметрия:", df[target].skew())
    print("Эксцесс:", df[target].kurtosis())

if target is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[target], kde=True, ax=axes[0])
    axes[0].set_title(f"Распределение {target}")
    sns.boxplot(x=df[target], ax=axes[1])
    axes[1].set_title(f"Ящик с усами: {target}")
    plt.tight_layout()
    plt.show()

    print("Асимметрия:", df[target].skew())
    print("Эксцесс:", df[target].kurtosis())

In [ ]:
for col in num_cols:
    if col == target:
        continue
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f"Гистограмма: {col}")
    plt.show()

Гистограммы показывают форму распределения. Например, size может быть скошена вправо, total_floor_count — иметь редкие большие значения, а tom — быть сосредоточенным в узком диапазоне

In [ ]:
if target is not None:
    for col in num_cols:
        if col == target:
            continue
        plt.figure(figsize=(6, 4))
        sns.scatterplot(data=df, x=col, y=target, alpha=0.3)
        plt.title(f"{col} и {target}")
        plt.show()

Диаграммы рассеяния показывают направление связи. Ожидается, что size сильнее всего положительно связан с ценой. Для building_age связь может быть отрицательной. Наличие нелинейности и выбросов влияет на выбор ансамблевых моделей

In [ ]:
for col in num_cols:
    if col == target:
        continue
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Ящик с усами: {col}")
    plt.show()

Ящики с усами помогают увидеть выбросы. Для size и total_floor_count возможны экстремальные значения. Выбросы не удаляются автоматически: сначала оценивается их природа — ошибка ввода или реальный объект

In [ ]:
if len(num_cols) > 1:
    corr = df[num_cols].corr()
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Тепловая карта корреляций")
    plt.show()

Тепловая карта показывает линейные связи между числовыми признаками и целевой переменной. Сильная корреляция признаков между собой указывает на мультиколлинеарность. Для линейных моделей это критично, для ансамблевых — менее критично, но всё равно учитывается

In [ ]:
if target is not None:
    for col in cat_cols:
        top_categories = df[col].value_counts().head(10).index
        temp = df[df[col].isin(top_categories)]
        plt.figure(figsize=(8, 4))
        sns.barplot(data=temp, x=col, y=target, estimator=np.mean, errorbar=None)
        plt.xticks(rotation=45)
        plt.title(f"Средняя {target} по {col}")
        plt.show()

Средняя цена по категориям показывает, какие категории различаются сильнее всего. Например, Satılık и Kiralık могут иметь принципиально разный уровень цен. Это подтверждает необходимость кодирования категориальных признаков

In [ ]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = missing["missing_count"] / len(df) * 100
missing = missing.sort_values("missing_count", ascending=False)

print("Пропуски по столбцам:")
print(missing)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing.index, y=missing["missing_pct"])
plt.xticks(rotation=90)
plt.ylabel("Доля пропусков, %")
plt.title("Пропуски по столбцам")
plt.show()

print("Дубликаты строк:", df.duplicated().sum())

Пропуски могут быть в building_age, total_floor_count, room_count, size. Стратегия обработки зависит от доли пропусков и смысла признака. Дубликаты объявлений могут искажать обучение и требуют удаления

In [ ]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()
    return count, lower, upper

for col in num_cols:
    if col == target:
        continue
    count, low, high = iqr_outlier_count(df[col].dropna())
    print(f"{col}: выбросов = {count} | нижняя граница = {low:.2f}, верхняя = {high:.2f}")

Правило IQR даёт количественную оценку выбросов. Для площади и количества этажей выбросы могут быть реальными (большие объекты), поэтому решение об удалении или ограничении принимается отдельно

##   выводы

1. `size` — наиболее вероятный сильный положительный признак для цены.
2. `room_count` и `total_floor_count` положительно связаны с ценой, но слабее, чем площадь.
3. `building_age` снижает цену: новые здания дороже.
4. `listing_type` принципиально разделяет данные: продажа и аренда имеют разный масштаб цен.
5. Категориальные признаки `type`, `sub_type`, `listing_type` требуют кодирования.
6. В данных возможны пропуски и выбросы, которые нужно обработать до обучения моделей.

# 2 Этап

In [ ]:
model_df = df.copy()

if target is not None:
    model_df = model_df[model_df[target].notna()].copy()
    print("Строк после удаления пропусков в целевой переменной:", model_df.shape)

Строки без целевой переменной не могут использоваться для обучения с учителем. Их удаление обязательно

In [ ]:
def parse_room_count(x):
    if pd.isna(x):
        return np.nan, np.nan
    x = str(x).replace(" ", "")
    if "+" in x:
        parts = x.split("+")
        try:
            return float(parts[0]) + float(parts[1]), float(parts[1])
        except ValueError:
            return np.nan, np.nan
    try:
        return float(x), np.nan
    except ValueError:
        return np.nan, np.nan

if "room_count" in model_df.columns:
    parsed = model_df["room_count"].apply(parse_room_count)
    model_df["total_rooms"] = [x[0] for x in parsed]
    model_df["living_rooms"] = [x[1] for x in parsed]

room_count хранится как 2+1, где первое число — комнаты, второе — гостиная. Создаются два числовых признака: общее количество комнат и количество гостиных. Это делает признак пригодным для моделей

In [ ]:
def parse_building_age(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().replace(" ", "")
    is_range = "arası" in x or "arasi" in x

    # Извлекаем все числа вручную, без re
    nums = []
    current = ""
    for ch in x:
        if ch.isdigit():
            current += ch
        else:
            if current:
                nums.append(float(current))
                current = ""
    if current:
        nums.append(float(current))

    if not nums:
        return np.nan
    if is_range and len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    return nums[0]

if "building_age" in model_df.columns:
    model_df["building_age_num"] = model_df["building_age"].apply(parse_building_age)

building_age может содержать диапазоны вида 5-10 и слово arası («между»). Для модели создаётся числовой признак building_age_num как середина диапазона. Извлечение чисел сделано без регулярных выражений — простым перебором символов

In [ ]:
for col in ["start_date", "end_date"]:
    if col in model_df.columns:
        model_df[col] = pd.to_datetime(model_df[col], errors="coerce")

if "start_date" in model_df.columns and "end_date" in model_df.columns:
    model_df["listing_days"] = (model_df["end_date"] - model_df["start_date"]).dt.days

if "start_date" in model_df.columns:
    model_df["start_year"] = model_df["start_date"].dt.year
    model_df["start_month"] = model_df["start_date"].dt.month
    model_df["start_dow"] = model_df["start_date"].dt.dayofweek

Из дат извлекаются длительность размещения объявления, год, месяц и день недели. Эти признаки могут отражать сезонность и активность рынка

In [ ]:
drop_cols = []
for col in ["start_date", "end_date", "room_count", "building_age"]:
    if col in model_df.columns:
        drop_cols.append(col)

model_df = model_df.drop(columns=drop_cols, errors="ignore")

Исходные столбцы, из которых уже извлечена информация, удаляются, чтобы не дублировать признаки и не создавать шум

In [ ]:
if target is not None:
    drop_for_X = [target]
    feature_cols = [c for c in model_df.columns if c not in drop_for_X]

    numeric_features = [
        c for c in feature_cols
        if pd.api.types.is_numeric_dtype(model_df[c])
    ]
    categorical_features = [
        c for c in feature_cols
        if c not in numeric_features
    ]

    print("Числовые признаки:", numeric_features)
    print("Категориальные признаки:", categorical_features)

Признаки делятся на числовые и категориальные. Для числовых применяется медианная импутация и масштабирование, для категориальных — модальная импутация и One-Hot Encoding

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Пропуски в числовых признаках заполняются медианой — она устойчива к выбросам. Пропуски в категориальных — модой. One-Hot Encoding создаёт бинарные столбцы для категорий. Масштабирование обязательно для линейных моделей и полезно для ансамблевых

In [ ]:
if target is not None:
    X = model_df[feature_cols]
    y = model_df[target]

    # 70% train, 15% validation, 15% test
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
    )

    print("Train:", X_train.shape, y_train.shape)
    print("Validation:", X_val.shape, y_val.shape)
    print("Test:", X_test.shape, y_test.shape)

    # Обучение препроцессора только на train — защита от утечки данных
    X_train_prep = preprocessor.fit_transform(X_train)
    X_val_prep = preprocessor.transform(X_val)
    X_test_prep = preprocessor.transform(X_test)

    print("Размеры после предобработки:")
    print(X_train_prep.shape, X_val_prep.shape, X_test_prep.shape)

Данные делятся на train/validation/test в пропорции 70/15/15. Препроцессор обучается только на train, чтобы избежать утечки данных. Все преобразования для validation и test применяются на основе статистик train

In [ ]:
if target is not None:
    with open("preprocessor.pkl", "wb") as f:
        pickle.dump(preprocessor, f)

    X_train.to_csv("X_train_raw.csv", index=False)
    X_val.to_csv("X_val_raw.csv", index=False)
    X_test.to_csv("X_test_raw.csv", index=False)

    y_train.to_csv("y_train.csv", index=False)
    y_val.to_csv("y_val.csv", index=False)
    y_test.to_csv("y_test.csv", index=False)

    print("Сохранено: preprocessor.pkl и файлы train/val/test.")

Сохраняются препроцессор и разделённые выборки. Это позволяет другому участнику обучить ансамблевые модели без повторной предобработки и без риска утечки данных. Для сохранения используется стандартный pickle

In [ ]:
if target is not None:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train_prep, y_train)

    pred_val = rf.predict(X_val_prep)

    mae = mean_absolute_error(y_val, pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred_val))
    r2 = r2_score(y_val, pred_val)

    print("Проверка RandomForest на validation")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

Это быстрая проверка, что после предобработки данные пригодны для ансамблевых моделей. Импорт RandomForestRegressor и метрик сделан локально, чтобы не засорять основной блок импортов. Основной подбор гиперпараметров и сравнение моделей выполняются на этапе 3

# ВСЁ, ТЫ ОТСЮДА ПРОДОЛЖАЕШЬ